# Пара 1. Бунин: частоты и графики (демо-корпус)

**Материал:** «Кавказ», «Холодная осень», «Чистый понедельник»  
**Зачем:** освоить общий пайплайн на знакомом корпусе курса, прежде чем переносить его на тексты диссертации.


## Старт в Colab
1. **Файл → Сохранить копию на Диске** (чтобы правки не пропали).
2. Справа сверху — **Подключить**.
3. Ячейки сверху вниз: Play или Shift+Enter.
Базовый Colab **бесплатный**. Правило занятия: **вопрос → процедура → картинка → цитата → ограничение**.


## 1. Загрузка трёх рассказов


In [ ]:
import re
import requests
import pandas as pd
import matplotlib.pyplot as plt

BASE = "https://pavelaleks.github.io/samarkand_lectures/demo-corpora/bunin-temnye-allei/txt/"
STORIES = {
    "Кавказ": "kavkaz.txt",
    "Холодная осень": "holodnaya-osen.txt",
    "Чистый понедельник": "chistyj-ponedelnik.txt",
}
texts = {}
for title, fname in STORIES.items():
    r = requests.get(BASE + fname, timeout=60)
    r.raise_for_status()
    r.encoding = "utf-8"
    texts[title] = r.text
    print(f"{title}: ~{len(texts[title].split())} слов")


## 2. Токенизация и поля любовь / память / время


In [ ]:
def tokenize(text):
    return re.findall(r"[А-Яа-яЁёA-Za-z]+", text.lower())

FIELDS = {
    "любовь": {"любов", "любил", "любила", "любит", "страсть", "сердц"},
    "память": {"памят", "вспомн", "припомин", "забыл", "забыт"},
    "время": {"год", "лет", "вечер", "ночь", "утро", "осень", "зима", "весна", "лето"},
}

def field_hits(tokens, stems):
    return sum(1 for t in tokens if any(t.startswith(s) for s in stems))

rows = []
for title, text in texts.items():
    toks = tokenize(text)
    n = max(len(toks), 1)
    row = {"рассказ": title, "слов": n}
    for name, stems in FIELDS.items():
        c = field_hits(toks, stems)
        row[name] = c
        row[f"{name}_на_1000"] = round(1000 * c / n, 2)
    rows.append(row)

df = pd.DataFrame(rows)
df


## 3. Столбцы + Zipf одного рассказа


In [ ]:
ax = df.set_index("рассказ")[[c for c in df.columns if c.endswith("_на_1000")]].plot(
    kind="bar", figsize=(9, 4), rot=15
)
ax.set_ylabel("на 1000 слов")
ax.set_title("Смысловые поля (учебный словарь)")
plt.tight_layout()
plt.show()

from collections import Counter
toks = tokenize(texts["Чистый понедельник"])
ranks = sorted(Counter(toks).values(), reverse=True)[:80]
plt.figure(figsize=(7, 4))
plt.loglog(range(1, len(ranks) + 1), ranks, marker="o", linestyle="none", markersize=3)
plt.xlabel("ранг"); plt.ylabel("частота"); plt.title("Zipf · Чистый понедельник")
plt.tight_layout(); plt.show()


## Сдать с пары
1 график + 1 цитата, которая **поддерживает или ломает** пик поля. Словарь учебный — это ограничение, не «истина Бунина».
